# 01 · Exploratory analysis

Working off the warehouse built by `run_pipeline.py` (at least `--steps load`).
Goal here is to understand *who* leaves and *when*, before any modelling.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from churn.analysis import style
from churn.pipeline.load import read_query

style.apply()
pd.set_option("display.max_columns", 50)
%matplotlib inline

In [ ]:
df = read_query("SELECT * FROM vw_feature_mart")
print(df.shape)
df.head()

In [ ]:
df.describe(include="number").T.round(2)

## Target balance
Roughly one in four customers churned in the snapshot, so accuracy alone would be misleading.
PR-AUC and lift are more useful later on.

In [ ]:
df["churn"].value_counts(normalize=True).rename({0: "retained", 1: "churned"}).to_frame("share")

## Churn by the obvious suspects

In [ ]:
cats = ["contract_type", "internet_service", "payment_method", "paperless_billing",
        "senior_citizen", "family", "tech_support", "online_security"]
summary = []
for c in cats:
    g = df.groupby(c)["churn"].agg(rate="mean", n="size").reset_index().rename(columns={c: "level"})
    g.insert(0, "feature", c)
    summary.append(g)
pd.concat(summary).assign(rate=lambda d: d["rate"].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x="tenure", hue="churn", bins=36, stat="density", common_norm=False,
             element="step", palette={0: style.BLUE, 1: style.ORANGE}, ax=ax)
style.titled(ax, "Tenure by outcome", "most churn happens in the first year")
plt.show()

## Is it the price or the product?
Fiber customers churn a lot - but they also pay the most. Compare within the same product and contract
using `price_vs_peers` (monthly charge minus the average for their internet × contract group).

In [ ]:
df["price_quartile"] = pd.qcut(df["price_vs_peers"], 4, labels=["cheapest", "q2", "q3", "priciest"])
pd.crosstab([df["contract_type"], df["internet_service"]], df["price_quartile"],
            values=df["churn"], aggfunc="mean").round(3)

## Statistical checks
Chi-square for the categorical drivers and Mann-Whitney for the numeric ones. With ~7k rows almost
everything is "significant", so the effect size (Cramér's V / rank-biserial) is what matters.

In [ ]:
from scipy import stats

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, p, _, _ = stats.chi2_contingency(ct)
    n = ct.to_numpy().sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1))), p

rows = []
for c in ["contract_type", "internet_service", "payment_method", "tech_support", "online_security",
          "paperless_billing", "senior_citizen", "gender", "partner", "streaming_tv"]:
    v, p = cramers_v(df[c], df["churn"])
    rows.append({"feature": c, "cramers_v": round(v, 3), "p_value": p})
pd.DataFrame(rows).sort_values("cramers_v", ascending=False)

In [ ]:
rows = []
for c in ["tenure", "monthly_charges", "total_charges", "n_services", "charge_drift", "price_vs_peers"]:
    a, b = df.loc[df.churn == 1, c], df.loc[df.churn == 0, c]
    u, p = stats.mannwhitneyu(a, b)
    rows.append({"feature": c, "median_churned": a.median(), "median_retained": b.median(),
                 "rank_biserial": round(1 - 2 * u / (len(a) * len(b)), 3), "p_value": p})
pd.DataFrame(rows)

## Retention curves

In [ ]:
from churn.analysis.eda import retention_curves
from churn.pipeline.transform import transform
from churn.pipeline.extract import extract

clean = transform(extract())
fig, km = retention_curves(clean)
plt.show()
km.round(3)

## Notes going into modelling
- Contract type dominates everything else. Month-to-month customers carry most of the churn.
- Tenure effect is strongest in the first 6-12 months, then flattens.
- Electronic check + paperless billing is a risky combo, probably a proxy for lower engagement.
- Security / tech-support add-ons go with much lower churn; streaming add-ons don't.
- `gender` shows no relationship at all - kept in the model only as a sanity check.